# SavorLink Strategy III: Multi-Modal Cold-Start Food Pairing
## High-Performance GPU Training & Multi-Seed Benchmark Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Course / Project**: Applied Foundations of Machine Learning (AFML) — SavorLink Strategy III  
**Target Hardware**: Google Colab Free-Tier Tesla T4 GPU (CUDA Accelerated)  

### What This Notebook Does:
This notebook handles **all GPU-intensive and VRAM-intensive deep learning workloads**:
1. Loads the lightweight precomputed bundle (`colab_training_bundle.zip`) containing pre-extracted features and similarities for all 4 modalities (Molecular, Semantic, Hybrid, Taxonomic).
2. Trains the Inductive Dual-Branch GNN with **Pairwise BPR (Bayesian Personalized Ranking) Loss** on CUDA.
3. Executes full multi-seed benchmarks (Seeds 42, 123, 456) across all 4 modalities (12 deep learning runs).
4. Performs GPU-accelerated all-pairs candidate ranking in parallel.
5. Outputs the publication-grade $4 \times 2$ comparative table and bundles results for 1-click download.

--- 
## 1. Hardware Verification & GPU Environment Setup

In [ ]:
import torch
import os
import sys

print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU:      {gpu_name} ({vram_gb:.2f} GB VRAM)")
    device = torch.device("cuda")
else:
    print("WARNING: CUDA not detected! In Colab, go to: Runtime -> Change runtime type -> T4 GPU")
    device = torch.device("cpu")
print("=" * 60)

# Install lightweight requirements
%pip install pyyaml scikit-learn scipy tqdm --quiet


--- 
## 2. Unzip Precomputed Training Bundle
Upload `colab_training_bundle.zip` (generated locally by `src/export_colab_bundle.py`, ~15 MB) to Colab files.

In [ ]:
import os, sys, zipfile

bundle_path = "colab_training_bundle.zip"
if not os.path.exists(bundle_path) and os.path.exists("/content/colab_training_bundle.zip"):
    bundle_path = "/content/colab_training_bundle.zip"

if os.path.exists(bundle_path):
    print(f"Found bundle: {bundle_path} ({os.path.getsize(bundle_path) / 1e6:.2f} MB). Unzipping...")
    with zipfile.ZipFile(bundle_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Unzipped bundle successfully!")
else:
    print("Upload colab_training_bundle.zip in the Colab file browser on the left, then rerun this cell.")

# Ensure project root is in sys.path
if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))


--- 
## 3. Verify Precomputed Features Across the 4 Modalities

In [ ]:
import numpy as np
import pickle
from pathlib import Path

modalities = {
    "molecular": {"feat": "data/cache/node_features.npy", "sim": "data/cache/topk_similarities.pkl"},
    "semantic":  {"feat": "data/cache/semantic_features.npy", "sim": "data/cache/semantic_similarities.pkl"},
    "hybrid":    {"feat": "data/cache/hybrid_features.npy", "sim": "data/cache/hybrid_similarities.pkl"},
    "taxonomic": {"feat": "data/cache/taxonomic_features.npy", "sim": "data/cache/taxonomic_similarities.pkl"},
}

print("Modality Feature Audit:")
for mod, paths in modalities.items():
    f_ok = os.path.exists(paths["feat"])
    s_ok = os.path.exists(paths["sim"])
    f_shape = np.load(paths["feat"]).shape if f_ok else "MISSING"
    print(f"  [{mod.upper():<10}] Features: {str(f_shape):<16} | Similarities Cached: {s_ok}")


--- 
## 4. GPU Multi-Modal GNN Training Suite (Pairwise BPR Loss)
Runs full training across all 4 modalities and multiple random seeds on CUDA.

In [ ]:
import subprocess

seeds = [42, 123, 456]
seeds_str = ",".join(map(str, seeds))

results_log = {}
for mod in ["molecular", "semantic", "hybrid", "taxonomic"]:
    print("=" * 65)
    print(f"  TRAINING GNN ON GPU — MODALITY: {mod.upper()}")
    print("=" * 65)
    cmd = [
        sys.executable, "src/train.py",
        "--config", "configs/default.yaml",
        "--modality", mod,
        "--seeds", seeds_str,
        "--loss", "bpr",
        "--epochs", "40",
    ]
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-1500:] if len(p.stdout) > 1500 else p.stdout)
    if p.returncode != 0:
        print(f"[ERROR in {mod}]:\n{p.stderr}")


--- 
## 5. Unified Comparative Results & Publication Table ($4 \times 2$ Matrix)

In [ ]:
# Load and display final benchmark table
import pandas as pd
report_path = Path("jackfruit/reports/versioned_reports.md")
if not report_path.exists():
    report_path = Path("reports/versioned_reports.md")

if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        text = f.read()
    print("=" * 70)
    print("  LATEST BENCHMARK REPORT TABLE:")
    print("=" * 70)
    # Print the last table in the report
    tables = [s for s in text.split("## ") if "Benchmark" in s or "Results" in s]
    if tables:
        print(tables[-1])
    else:
        print(text[-2000:])


--- 
## 6. Export Results & Trained Checkpoints

In [ ]:
import shutil
from google.colab import files

out_zip = "savorlink_colab_results.zip"
!zip -r -q {out_zip} data/exports/ data/cache/*similarities.pkl reports/
print(f"Bundled results into {out_zip} ({os.path.getsize(out_zip) / 1e6:.2f} MB)")

# Trigger 1-click browser download in Colab
files.download(out_zip)
